In [1]:
%%capture
! pip install -U evaluate transformers accelerate gdown

In [2]:
import pandas as pd
import numpy as np

# Tokenization utilities
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup,
                          Trainer,
                          TrainingArguments)

from huggingface_hub import login

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split, DataLoader

# Sklearn
import sklearn
from sklearn.utils.class_weight import compute_class_weight

# Evaluate
import evaluate

# Runtime utilities
import math
import time
from tqdm import tqdm, trange
import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'

2025-07-31 20:07:07.688559: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753992428.085469      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753992428.197341      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
# import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
login(token='')

In [6]:
class ModelFinetuner:
    """
    A class for fine-tuning the DepRoBERTa model on a social media dataset for stigma detection task.
    """

    def __init__(self, file_path):
        """
        Initialize the ModelFinetuner class.

        Args:
            file_path (str): The path to the JSON file containing the dataset.
        """
        self.file_path = file_path

        self.accuracy = evaluate.load('accuracy')
        self.precision = evaluate.load('precision')
        self.recall = evaluate.load('recall')
        self.f1 = evaluate.load('f1')

    def load_dataset(self):
        """
        Load the dataset from the JSON file.
        """
        with open(f'{self.file_path}/train_data.json', 'r') as f:
          self.train_data = json.loads(f.read())
          f.close()

        with open(f'{self.file_path}/val_data.json', 'r') as f:
          self.val_data = json.loads(f.read())
          f.close()


    def create_dataset(self):
        """
        Split the dataset into train, validation, and test sets.

        Args:
            test_size (float): The proportion of the dataset to include in the test split.
            val_size (float): The proportion of the dataset to include in the validation split.
        """

        self.load_dataset()

        train_txt, train_label = self.format_dataset(self.train_data)
        val_txt, val_label = self.format_dataset(self.val_data)

        self.tokenizer = AutoTokenizer.from_pretrained("rafalposwiata/deproberta-large-depression", do_lower_case=True)

        train_encodings = self.tokenizer(train_txt, truncation=True, padding=True)
        val_encodings = self.tokenizer(val_txt, truncation=True, padding=True)

        self.train_dataset = PostDataset(train_encodings, train_label)
        self.val_dataset = PostDataset(val_encodings, val_label)


    def format_dataset(self, data):
        """
        Create a PyTorch dataset from the given encodings and labels.

        Args:
            encodings (dict): The tokenized input encodings.
            labels (list): The corresponding labels.

        Returns:
            IMDbDataset: A PyTorch dataset object.
        """
        sentences = []
        labels = []

        for post in data:
          title, post_text = post['title'], post['post text']
          if title is None:
            title = ''
          if post_text is None:
            post_text = ''

          if post['label'] == 'no stigma' or post['label'] == 'unclear':
            labels.append(0)
            sentences.append(f'{title}-{post_text}')
          elif post['label'] == 'yes stigma':
            labels.append(1)
            sentences.append(f'{title}-{post_text}')

        return (sentences, labels)


    def get_class_weight(self):
      _, train_label = self.format_dataset(self.train_data)
      classes = np.unique(train_label)

      class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_label)

      # Convert to torch tensor
      class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
      return class_weights_tensor


    def fine_tune(self, epochs=5, batch_size=16, warmup_steps=500, weight_decay=0.01):
        """
        Fine-tune the DepRoBERTa model on the training data.

        Args:
            epochs (int): The number of training epochs.
            batch_size (int): The batch size for training.
            warmup_steps (int): The number of warmup steps for the learning rate scheduler.
            weight_decay (float): The strength of weight decay regularization.
        """
        training_args = TrainingArguments(
            output_dir='./results',
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            warmup_steps=warmup_steps,
            weight_decay=weight_decay,
            logging_dir='./logs',
            logging_steps=10,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=10,
            load_best_model_at_end=True,
            metric_for_best_model='f1',
            report_to="none"
            )

        self.model = AutoModelForSequenceClassification.from_pretrained(
            "rafalposwiata/deproberta-large-depression",
            output_attentions = False,
            output_hidden_states = False,
            )

        self.model.classifier.out_proj = nn.Linear(1024, 2)
        self.model.num_labels = 2

        for param in self.model.base_model.parameters():
            param.requires_grad = False

        # Unfreeze the classification head
        for param in self.model.classifier.parameters():
            param.requires_grad = True

        self.model.cuda()

        self.trainer = WeightedLossTrainer(
            model=self.model,
            args=training_args,
            train_dataset=self.train_dataset,
            eval_dataset=self.val_dataset,
            compute_metrics=self.compute_metrics,
        )

        self.trainer.train()


    def compute_metrics(self, pred):
        """
        Compute evaluation metrics based on the predictions.

        Args:
            pred (EvalPrediction): The model's predictions.

        Returns:
            dict: A dictionary containing the computed metrics.
        """
        predictions, labels = pred
        predictions = np.argmax(predictions, axis=1)
        result = {'accuracy': self.accuracy.compute(predictions=predictions, references=labels)['accuracy'],
              'precision': self.precision.compute(predictions=predictions, references=labels, average = 'macro')['precision'],
              'recall': self.recall.compute(predictions=predictions, references=labels, average = 'macro')['recall'],
              'f1': self.f1.compute(predictions=predictions, references=labels, average = 'macro')['f1']}
        return result

    def evaluate_model(self):
        """
        Evaluate the fine-tuned model on the test set.
        """
        print(self.trainer.evaluate(self.test_dataset))

    def save_model(self, model_name):
        """
        Save the fine-tuned model and tokenizer to the Hugging Face Hub.

        Args:
            model_name (str): The name of the model on the Hugging Face Hub.
        """
        repo_name = f'haji80mr-uoft/{model_name}'
        self.tokenizer.push_to_hub(repo_name)
        self.model.push_to_hub(repo_name)



class PostDataset(torch.utils.data.Dataset):
    """
    A PyTorch dataset for the movie genre classification task.
    """

    def __init__(self, encodings, labels):
        """
        Initialize the IMDbDataset class.

        Args:
            encodings (dict): The tokenized input encodings.
            labels (list): The corresponding labels.
        """
        # TODO: Implement initialization logic

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        """
        Get a single item from the dataset.

        Args:
            idx (int): The index of the item to retrieve.

        Returns:
            dict: A dictionary containing the input encodings and labels.
        """
        # TODO: Implement item retrieval logic
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        """
        Get the length of the dataset.

        Returns:
            int: The number of items in the dataset.
        """
        # TODO: Implement length computation logic
        return len(self.labels)

In [7]:
finetuner = ModelFinetuner('/kaggle/working')
finetuner.load_dataset()

class_weights = finetuner.get_class_weight()
loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(device))

In [8]:
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, num_items_in_batch = 8, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [9]:
finetuner.create_dataset()

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [10]:
finetuner.fine_tune(epochs=50)

config.json:   0%|          | 0.00/924 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.749400,0.633106,0.927152,0.466667,0.496454,0.481100
2,0.674400,0.621100,0.754967,0.559066,0.682979,0.549326
3,0.630500,0.585046,0.728477,0.586935,0.808156,0.568180
4,0.610500,0.553417,0.682119,0.575094,0.783333,0.534669
5,0.572500,0.512037,0.715232,0.583235,0.801064,0.558329
6,0.513600,0.454862,0.788079,0.584179,0.747163,0.589674
7,0.590800,0.421676,0.781457,0.581437,0.743617,0.584230
8,0.504700,0.442844,0.761589,0.597600,0.825887,0.594086
9,0.499700,0.373694,0.860927,0.628095,0.786170,0.660674
10,0.447300,0.431958,0.728477,0.586935,0.808156,0.568180


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/modul

In [11]:
finetuner.trainer.evaluate(finetuner.val_dataset)

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 0.3484193682670593,
 'eval_accuracy': 0.9139072847682119,
 'eval_precision': 0.6946883230904302,
 'eval_recall': 0.8145390070921985,
 'eval_f1': 0.7356228956228956,
 'eval_runtime': 7.5886,
 'eval_samples_per_second': 19.898,
 'eval_steps_per_second': 0.659,
 'epoch': 50.0}

In [12]:
with open(f'{finetuner.file_path}/test_data.json', 'r') as f:
          test_data = json.loads(f.read())
          f.close()

test_txt, test_label = finetuner.format_dataset(test_data)

test_encodings = finetuner.tokenizer(test_txt, truncation=True, padding=True)

test_dataset = PostDataset(test_encodings, test_label)

In [13]:
finetuner.trainer.evaluate(test_dataset)

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 0.5777804255485535,
 'eval_accuracy': 0.9006622516556292,
 'eval_precision': 0.6887382690302398,
 'eval_recall': 0.7017837235228539,
 'eval_f1': 0.694949494949495,
 'eval_runtime': 15.5657,
 'eval_samples_per_second': 19.402,
 'eval_steps_per_second': 0.642,
 'epoch': 50.0}

In [14]:
finetuner.save_model('DepRoberta_finetuned_V1_kaggle')

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmpq3_t7ydy/model.safetensors    :   0%|          | 5.52MB / 1.42GB            